In [1]:
import torch
import time
import arc_compressor_batch as arc_compressor
import preprocessing
import solution_selection
# import visualization  # Uncomment if you want to use visualization.plot_problem
from train import take_step  # Assuming take_step is defined in train.py

In [2]:
# %debug
task_nums = [0,1,2]
split = "training"  # "training", "evaluation, or "test"

# Preprocess all tasks, make models, optimizers, and loggers. Make plots.
tasks = preprocessing.preprocess_tasks(split, task_nums)
models = []
optimizers = []
train_history_loggers = []
for task in tasks:
    model = arc_compressor.ARCCompressor(task)
    models.append(model)
    optimizer = torch.optim.Adam(model.weights_list, lr=0.01, betas=(0.5, 0.9))
    optimizers.append(optimizer)
    train_history_logger = solution_selection.Logger(task)
    # visualization.plot_problem(train_history_logger)
    train_history_loggers.append(train_history_logger)


In [3]:
model = models[0]
logits, x_mask, y_mask, KL_amounts, KL_names, = model.forward()

torch.Size([8, 3, 7, 6, 6, 16]) torch.Size([8, 16, 2]) torch.Size([8, 1, 1, 1, 1, 2])


In [4]:
tasks[0].problem.shape

torch.Size([3, 6, 6, 2])

In [6]:
logits.shape, x_mask.shape, y_mask.shape

(torch.Size([8, 8, 3, 7, 6, 6, 2]),
 torch.Size([8, 3, 6, 2]),
 torch.Size([8, 3, 6, 2]))

In [3]:
# %debug
task_stats = []

# Get the solution hashes so that we can check for correctness
true_solution_hashes = [task.solution_hash for task in tasks]

# Train the models one by one
for i, (task, model, optimizer, train_history_logger) in enumerate(zip(tasks, models, optimizers, train_history_loggers)):
    n_iterations = 1

    task_start_time = time.time()

    for train_step in range(n_iterations):
        track_last = (train_step == n_iterations - 1)
        take_step(task, model, optimizer, train_step, train_history_logger, track_last=track_last)

RuntimeError: Expected target size [1, 2, 2, 6, 2], got [1, 2, 2]